In [1]:
# %matplotlib inline
%config InlineBackend.figure_format = "retina"
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
})

import seaborn as sns
sns.set_theme(context="talk", style="whitegrid", 
              palette="colorblind", color_codes=True, 
              rc={"figure.figsize": [12, 8]})

import yfinance as yf
import numpy as np
import pandas as pd
import quantstats as qs

import requests
import csv

from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

import statsmodels.api as sm

from pypfopt import black_litterman
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier

In [2]:
# Carteira teorica do Idiv

url = "https://raw.githubusercontent.com/BDonadelli/Finance-playground/main/data/Cart_Idiv.csv"

response = requests.get(url)
response.encoding = 'utf-8'  # ou 'latin-1' se necessário

# Divide o conteúdo em linhas
linhas = response.text.splitlines()

# Ignora as duas primeiras e duas últimas linhas
linhas_filtradas = linhas[2:-2]

# Extrai a primeira coluna de cada linha restante
ASSETS = []
for linha in linhas_filtradas:
    # O separador é ";", pega o primeiro campo
    campos = linha.split(';')
    if campos:  # garante que a linha não está vazia
        ASSETS.append((campos[0],campos[4]))

ASSETS = [(ticker, float(str(value).replace(',', '.'))) for ticker, value in ASSETS]

try:
    pesos_indice = [value for ticker, value in ASSETS]
    tickers = [ticker+'.SA' for ticker, value in ASSETS]
    tickers.sort()
except :    
    tickers = [ticker+'.SA' for ticker in ASSETS]

tickers.append('AMBP3.SA')
tickers.sort()

print(tickers)

['ABCB4.SA', 'AGRO3.SA', 'ALOS3.SA', 'AMBP3.SA', 'BBAS3.SA', 'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BRAP4.SA', 'BRBI11.SA', 'BRSR6.SA', 'CMIG4.SA', 'CMIN3.SA', 'CPFE3.SA', 'CSMG3.SA', 'CURY3.SA', 'CXSE3.SA', 'DIRR3.SA', 'EGIE3.SA', 'EVEN3.SA', 'EZTC3.SA', 'FESA4.SA', 'FLRY3.SA', 'GRND3.SA', 'ISAE4.SA', 'ITSA4.SA', 'ITUB3.SA', 'ITUB4.SA', 'JHSF3.SA', 'KEPL3.SA', 'KLBN11.SA', 'LAVV3.SA', 'LEVE3.SA', 'LOGG3.SA', 'MBRF3.SA', 'ODPV3.SA', 'PETR3.SA', 'PETR4.SA', 'PGMN3.SA', 'POMO4.SA', 'RANI3.SA', 'RECV3.SA', 'SAPR11.SA', 'SLCE3.SA', 'SYNE3.SA', 'TAEE11.SA', 'TGMA3.SA', 'TIMS3.SA', 'UNIP6.SA', 'VALE3.SA', 'VBBR3.SA', 'VLID3.SA', 'VULC3.SA']


In [3]:
ASSETS = [
    "ABEV3.SA",
    "VLID3.SA",
    "OFSA3.SA",
    "TECN3.SA",
    "SOND5.SA",
    "MDNE3.SA",
    "CURY3.SA",
    "RANI3.SA",
    "JHSF3.SA",
    "LPSB3.SA",
    "MULT3.SA",
    "ITSA4.SA",
    "RECV3.SA",
    "CSUD3.SA",
    "LUXM4.SA",
    "FIQE3.SA",
    "BLAU3.SA",
    "SHUL4.SA",
    "EALT4.SA",
    "RSUL4.SA",
    "POMO4.SA",
    "PETR4.SA",
    "SBSP3.SA",
    "CSMG3.SA",
    "WIZC3.SA",
    "MILS3.SA",
    "CAMB3.SA",
    "VULC3.SA",
    "GRND3.SA"
]

#### Parâmetros

In [4]:
rf = 0.14               # taxa livre de risco
n_days=252              # dias no ano do calendario financeiro, assumindo dados diários pegos no Yahoo Finance
#n_monte_carlo = 10**6 # quantidade de carteiras na simulação

# Definição do período e download dos dados
data_inicio = '2018-01-01'
data_fim = '2026-04-30'

#### preços de fechamento

Baixa dados e limpa a base

In [5]:
prices = yf.download(tickers, start=data_inicio, end=data_fim , auto_adjust=True)['Close']
prices.columns = [col.replace('.SA', '') for col in prices.columns]

benchm = yf.download('^BVSP', start=data_inicio, end=data_fim , auto_adjust=True)['Close']

[*********************100%***********************]  53 of 53 completed
[*********************100%***********************]  1 of 1 completed


In [6]:
# Empresas com mais de 'limiar' dados faltantes
limiar = 10
missing = prices.isna().sum()
empresas_missing = missing[missing > limiar].index.tolist()

print(missing[missing > limiar].sort_values(ascending=False))

tickers = [item for item in ASSETS if item not in empresas_missing]



BRBI11    872
RECV3     824
CXSE3     821
CMIN3     774
CURY3     674
LAVV3     663
PGMN3     663
AMBP3     625
ALOS3     394
LOGG3     242
VBBR3      49
dtype: int64


In [7]:
import plotly.express as px

if empresas_missing:
    fig = px.line(
        prices[empresas_missing].reset_index(),
        x='Date',
        y=empresas_missing,
        title='Empresas com mais de 10 dados faltantes',
        labels={'value': 'Preço', 'Date': 'Data', 'variable': 'Empresa'}
    )

    fig.show()

In [8]:
# mantem colunas (axis=1) ue possuem no mínimo len(prices) - 20 valores não-nulos.
prices = prices.dropna(axis=1, thresh=len(prices) - 10)
# preenche dados faltantes repetindo ultimo valor
prices = prices.ffill()

prices

,ABCB4,AGRO3,BBAS3,BBDC3,BBDC4,BBSE3,BRAP4,BRSR6,CMIG4,CPFE3,...,SAPR11,SLCE3,SYNE3,TAEE11,TGMA3,TIMS3,UNIP6,VALE3,VLID3,VULC3
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,9.208960,6.842369,9.456818,11.138399,12.179820,13.367787,4.202160,7.811340,1.498755,12.097425,...,12.398586,3.451246,1.566560,9.288569,11.868073,7.978171,5.058343,21.669312,11.916831,5.375476
2018-01-03,9.247839,6.836968,9.577430,11.185785,12.235813,13.377099,4.241884,7.848089,1.485834,11.947999,...,12.291392,3.481202,1.650874,9.301554,11.868073,7.984216,5.428318,21.539463,12.096827,5.641647
2018-01-04,9.220068,7.058387,9.669328,11.389475,12.436570,13.405048,4.362472,7.895335,1.468608,11.860830,...,12.192610,3.467472,1.720012,9.154390,12.324996,7.947953,5.370700,21.627762,12.096827,5.786304
2018-01-05,9.353373,7.204199,9.669328,11.392930,12.507012,13.493545,4.467455,7.998996,1.470761,11.667819,...,12.285089,3.532378,1.720012,9.197675,12.461480,8.014437,5.455612,21.965372,12.072000,5.786304
2018-01-08,9.442240,7.236601,9.692302,11.392930,12.503496,13.572727,4.545482,8.132941,1.475068,11.854608,...,12.211523,3.513655,1.736875,9.154390,12.461480,7.905646,5.701251,22.453606,12.040965,5.815235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,25.160000,19.680000,23.000000,17.191864,19.949961,34.270000,24.059999,16.155130,13.088566,50.148193,...,42.849998,17.530001,4.060000,43.116375,32.939999,26.049999,60.869999,85.970001,19.563784,16.120001
2026-04-24,25.280001,19.850000,22.700001,17.091970,19.900011,34.290001,23.950001,15.876423,12.804031,49.388653,...,43.610001,17.370001,4.000000,42.585388,32.230000,26.100000,60.540001,85.870003,19.652891,15.940000
2026-04-27,24.930000,19.139999,22.510000,16.952118,19.710201,33.950001,23.850000,15.687300,12.558743,48.906994,...,41.549999,17.240000,3.940000,42.102673,32.250000,25.770000,59.880001,85.500000,19.246962,15.780000


In [9]:
returns = prices.pct_change().dropna()

#log retorno para matriz de covriancia ledoit wolf, tomar cuidado com parametros das bibliotecas

print(len(prices),len(returns),len(tickers),len(ASSETS),len(prices.columns))

2068 2067 29 29 42


Matriz covariância por Ledoit Wolf

In [10]:
from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

"""
cov_matrix é uma NxN matriz de covariância
tickers é um dicionário com os ativos
"""

pesos = [1 / len(ASSETS)] * len(ASSETS)
# dicionario_pesos = dict(zip(tickers, pesos))

#cov = np.cov(market_prices)

model = LedoitWolf()
cov_matrix = model.fit(returns).covariance_
df_cov_matrix = pd.DataFrame(cov_matrix, index=returns.columns, columns=returns.columns)

print(df_cov_matrix.shape)
#print(dicionario_pesos)
print(tickers)

tickers_validos = list(df_cov_matrix.columns)

pesos = np.repeat(1/len(tickers_validos), len(tickers_validos))

dicionario_pesos = dict(zip(tickers_validos, pesos))



(42, 42)
['ABEV3.SA', 'VLID3.SA', 'OFSA3.SA', 'TECN3.SA', 'SOND5.SA', 'MDNE3.SA', 'CURY3.SA', 'RANI3.SA', 'JHSF3.SA', 'LPSB3.SA', 'MULT3.SA', 'ITSA4.SA', 'RECV3.SA', 'CSUD3.SA', 'LUXM4.SA', 'FIQE3.SA', 'BLAU3.SA', 'SHUL4.SA', 'EALT4.SA', 'RSUL4.SA', 'POMO4.SA', 'PETR4.SA', 'SBSP3.SA', 'CSMG3.SA', 'WIZC3.SA', 'MILS3.SA', 'CAMB3.SA', 'VULC3.SA', 'GRND3.SA']


In [11]:
delta = black_litterman.market_implied_risk_aversion(prices)
prior = black_litterman.market_implied_prior_returns(
    dicionario_pesos,
    delta,
    cov_matrix
)
print(delta)

ABCB4     1.777735
AGRO3     2.011665
BBAS3     1.300449
BBDC3     0.932728
BBDC4     0.966698
BBSE3     2.321506
BRAP4     2.204130
BRSR6     1.226214
CMIG4     2.666148
CPFE3     2.861778
CSMG3     2.605379
DIRR3     2.237331
EGIE3     2.913169
EVEN3     0.752133
EZTC3     0.630332
FESA4     1.167987
FLRY3     0.321384
GRND3     0.549470
ISAE4     3.358211
ITSA4     2.427837
ITUB3     2.534811
ITUB4     1.915718
JHSF3     1.851542
KEPL3     1.938175
KLBN11    1.258539
LEVE3     1.699175
MBRF3     1.171516
ODPV3     0.967243
PETR3     2.150693
PETR4     2.226632
POMO4     1.389336
RANI3     1.247864
SAPR11    2.085161
SLCE3     2.054291
SYNE3     0.870921
TAEE11    5.249977
TGMA3     1.234115
TIMS3     2.061365
UNIP6     2.270381
VALE3     1.759532
VLID3     0.749243
VULC3     1.335834
dtype: float64


/home/caio/Documentos/ic_26/.venv/lib/python3.12/site-packages/pypfopt/black_litterman.py:45: RuntimeWarning: If cov_matrix is not a dataframe, market cap index must be aligned to cov_matrix
  warnings.warn(


In [12]:
print(prior)

ABCB4     0.000307
AGRO3     0.000203
BBAS3     0.000289
BBDC3     0.000194
BBDC4     0.000203
BBSE3     0.000270
BRAP4     0.000312
BRSR6     0.000235
CMIG4     0.000534
CPFE3     0.000383
CSMG3     0.000447
DIRR3     0.000516
EGIE3     0.000338
EVEN3     0.000215
EZTC3     0.000176
FESA4     0.000180
FLRY3     0.000053
GRND3     0.000088
ISAE4     0.000350
ITSA4     0.000429
ITUB3     0.000415
ITUB4     0.000345
JHSF3     0.000476
KEPL3     0.000247
KLBN11    0.000106
LEVE3     0.000267
MBRF3     0.000205
ODPV3     0.000109
PETR3     0.000466
PETR4     0.000479
POMO4     0.000286
RANI3     0.000195
SAPR11    0.000314
SLCE3     0.000213
SYNE3     0.000163
TAEE11    0.000493
TGMA3     0.000279
TIMS3     0.000284
UNIP6     0.000419
VALE3     0.000247
VLID3     0.000161
VULC3     0.000277
dtype: float64


In [13]:
fatores_nefin = pd.read_csv('nefin_factors.csv')

# A coluna Date já existe, só converter e setar como índice
fatores_nefin['Date'] = pd.to_datetime(fatores_nefin['Date'])
fatores_nefin.set_index('Date', inplace=True)

# Reamostrar os fatores diários para mensais
#fatores_mensais = fatores_nefin.resample('ME').sum()
#print(fatores_mensais)

fatores_mensais = (1 + fatores_nefin).resample('ME').prod() - 1
print(fatores_mensais)
#verificar se a taxa e fatores estao sendo colocadas certas mensalmente

# Alinhar com os retornos
datas_comuns = returns.index.intersection(fatores_mensais.index)
retornos_alinhados = returns.loc[datas_comuns]
fatores_alinhados = fatores_mensais.loc[datas_comuns]

                     Unnamed: 0  Rm_minus_Rf       SMB       HML       WML  \
Date                                                                         
2001-01-31 -1250660718674968577     0.139540  0.163653  0.147510 -0.012725   
2001-02-28   231432332370247679    -0.085317  0.052359  0.022571  0.071785   
2001-03-31 -2890597822505680897    -0.077331 -0.014624  0.060201  0.076311   
2001-04-30  7254507651604676607     0.026857 -0.120024 -0.155679 -0.049503   
2001-05-31  8983196893043490815    -0.003461 -0.094637 -0.154272 -0.017923   
...                         ...          ...       ...       ...       ...   
2025-12-31 -8644949316997480449     0.004598 -0.031421  0.009458 -0.028194   
2026-01-31  4353516797392060415     0.103267 -0.014013  0.043407  0.023017   
2026-02-28  3190475155598606335     0.026488 -0.044621 -0.033540  0.007398   
2026-03-31 -8315929580154126337    -0.021942 -0.039934  0.004906 -0.021615   
2026-04-30             39181339     0.001203  0.005243  0.009763

In [14]:
print(pesos)

[0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952
 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952 0.02380952]


In [16]:
lista_tickers = tickers_validos

# -------------------------------------------------------
# ETAPA 1: OLS — regressão Fama-French por ativo
col_rf       = 'Risk_Free'
fatores_cols = [c for c in fatores_alinhados.columns if c != col_rf]
X_ols         = sm.add_constant(fatores_alinhados[fatores_cols])
medias_fatores = fatores_alinhados[fatores_cols].mean()

betas_dict = {}
for ticker in lista_tickers:
    Y      = retornos_alinhados[ticker] - fatores_alinhados[col_rf]
    modelo = sm.OLS(Y, X_ols).fit()
    betas_dict[ticker] = modelo.params

betas_df = pd.DataFrame(betas_dict).T

# ETAPA 2: Retornos estimados r̂_i (Eq. 12)
retornos_estimados = (
    betas_df['const']
    + betas_df[fatores_cols].dot(medias_fatores)
)

# ETAPA 3: Grid nxn
book_to_market = {}
for ticker in lista_tickers:
    info = yf.Ticker(ticker + '.SA').info
    pb   = info.get('priceToBook', None)
    book_to_market[ticker] = 1 / pb if pb else None
book_to_market = pd.Series(book_to_market).dropna()

# ✅ market_cap criado ANTES de filtrar
market_cap = pd.Series(
    {ticker: peso for ticker, peso in zip(lista_tickers, pesos)}
)

# ✅ filtrar AMBOS para tickers com BM disponível
tickers_bm     = [t for t in lista_tickers if t in book_to_market.index]
market_cap     = market_cap[tickers_bm]      # ← corrigido (era market_cap[tickers_bm] sem definir antes)
book_to_market = book_to_market[tickers_bm]

# ✅ size_quintil calculado sobre market_cap já filtrado
n            = 3
size_quintil = pd.qcut(market_cap.rank(method='first'), n, labels=False)
bm_labels    = pd.Series(index=tickers_bm, dtype=int)

for g in range(n):
    grupo_tickers = size_quintil[size_quintil == g].index  # ← agora alinhado com tickers_bm
    bm_labels[grupo_tickers] = pd.qcut(
        book_to_market[grupo_tickers].rank(method='first'),
        n, labels=False, duplicates='drop'
    )

grupo_1  = [t for t in tickers_bm if size_quintil[t] == 0 and bm_labels[t] == n-1]
grupo_9 = [t for t in tickers_bm if size_quintil[t] == n-1 and bm_labels[t] == 0]

if not grupo_1 or not grupo_9:
    raise ValueError("Grupo 1 ou 25 vazio — verifique os dados de size/BM")


print(f"Grupo 1 (small/value): {grupo_1}")
print(f"Grupo 9 (big/growth):  {grupo_9}")

# ETAPA 4: P_t (Eq. 11)
P_t           = pd.Series(0.0, index=tickers_bm)
P_t[grupo_1] =  1 / len(grupo_1)
P_t[grupo_9] = -1 / len(grupo_9)

# ETAPA 5: q_t (Eq. 13-15)
q_t = retornos_estimados[grupo_1].mean() - retornos_estimados[grupo_9].mean()
print(f"\nView return q_t: {q_t:.6f}")

# print("Médias dos fatores (mensais):")
# print(medias_fatores)
# print(f"\nMédia Rf: {rf_media:.6f}")

# 2. Compare q_t com a magnitude típica dos retornos
print(f"\nq_t / média dos retornos estimados: {q_t / retornos_estimados.mean():.2%}")

# 3. Veja a dispersão de retornos_estimados
print(f"\nRetornos estimados FF:")
print(retornos_estimados.describe())
# ETAPA 6: Ω_t (Eq. 16)

# -------------------------------------------------------
# ETAPA 6: Incerteza da visão Ω_t (Eq. 16)  τ = 0.1
# -------------------------------------------------------
# tau     = 0.1
# Sigma   = df_cov_matrix.loc[tickers_bm, tickers_bm].values
# p1      = P_t.values.reshape(1, -1)
# omega_t = float(tau * (p1 @ Sigma @ p1.T))

# # -------------------------------------------------------
# # ETAPA 7: Retornos posteriores Black-Litterman (Eq. 7-8)
# # -------------------------------------------------------
# pi      = prior[tickers_bm].values.reshape(-1, 1)
# P       = p1
# q       = np.array([[q_t]])
# Omega   = np.array([[omega_t]])

# Sigma_BL = np.linalg.inv(
#     np.linalg.inv(tau * Sigma) + P.T @ np.linalg.inv(Omega) @ P
# )

# mu_BL = Sigma_BL @ (
#     np.linalg.inv(tau * Sigma) @ pi
#     + P.T @ np.linalg.inv(Omega) @ q
# )

# mu_BL_series = pd.Series(mu_BL.flatten(), index=tickers_bm)
# print(mu_BL_series.sort_values(ascending=False))



Grupo 1 (small/value): ['ABCB4', 'AGRO3', 'BBAS3', 'BRSR6', 'EVEN3']
Grupo 9 (big/growth):  ['POMO4', 'TGMA3', 'TIMS3', 'UNIP6', 'VULC3']

View return q_t: 0.000192

q_t / média dos retornos estimados: -32.32%

Retornos estimados FF:
count    42.000000
mean     -0.000596
std       0.000523
min      -0.001387
25%      -0.001022
50%      -0.000680
75%      -0.000257
max       0.000762
dtype: float64


In [19]:
tau     = 0.1
Sigma   = df_cov_matrix.loc[tickers_bm, tickers_bm].values
p1      = P_t.values.reshape(1, -1)


# ✅ Correto — .item() extrai o escalar de um array (1,1)
omega_t = (tau * (p1 @ Sigma @ p1.T)).item()

# Verificação de sanidade
print(f"q_t    : {q_t:.6f}")
print(f"omega_t: {omega_t:.6f}")
print(f"P_t não-zero: {P_t[P_t != 0]}")

# Preparar para BlackLittermanModel
P_matrix     = p1                        # (1, N)
Q_vector     = np.array([q_t])           # (1,)
Omega_matrix = np.array([[omega_t]])     # (1, 1)
cov_matrix_bm = df_cov_matrix.loc[tickers_bm, tickers_bm]
prior_bm      = prior[tickers_bm]

# Black-Litterman
bl = BlackLittermanModel(
    cov_matrix_bm,
    pi=prior_bm,
    Q=Q_vector,
    P=P_matrix,
    omega=Omega_matrix,
    tau=tau
)

mu_BL = bl.bl_returns()
print("\nRetornos posteriores BL:")
print(mu_BL.sort_values(ascending=False))

ef = EfficientFrontier(mu_BL, cov_matrix_bm, weight_bounds=(0, 0.10))
ef.max_sharpe()
weights = ef.clean_weights()
print("\nPesos ótimos (max Sharpe):")
print(pd.Series(weights).sort_values(ascending=False))

q_t    : 0.000192
omega_t: 0.000015
P_t não-zero: ABCB4    0.2
AGRO3    0.2
BBAS3    0.2
BRSR6    0.2
EVEN3    0.2
POMO4   -0.2
TGMA3   -0.2
TIMS3   -0.2
UNIP6   -0.2
VULC3   -0.2
dtype: float64

Retornos posteriores BL:
CMIG4     0.000539
DIRR3     0.000525
PETR4     0.000499
TAEE11    0.000491
PETR3     0.000486
JHSF3     0.000477
ITSA4     0.000454
CSMG3     0.000444
ITUB3     0.000437
CPFE3     0.000385
ITUB4     0.000375
ABCB4     0.000355
BBAS3     0.000352
ISAE4     0.000351
EGIE3     0.000339
UNIP6     0.000332
BRAP4     0.000308
SAPR11    0.000308
BRSR6     0.000294
EVEN3     0.000282
BBSE3     0.000281
LEVE3     0.000252
VALE3     0.000248
KEPL3     0.000246
AGRO3     0.000245
TIMS3     0.000240
BBDC4     0.000236
BBDC3     0.000225
TGMA3     0.000216
SLCE3     0.000215
VULC3     0.000208
MBRF3     0.000200
POMO4     0.000199
EZTC3     0.000195
RANI3     0.000193
FESA4     0.000171
SYNE3     0.000170
VLID3     0.000145
ODPV3     0.000110
KLBN11    0.000089
GRND3     0.000080
